In [31]:
import numpy as np
import math
import plotly.graph_objects as go

class magnet:
    def __init__(self, 
                 position:np.array = np.zeros((3,)), 
                 dipole:np.array = np.zeros((3,))):
        self.position = position
        self.dipole = dipole

def magnetFieldApprox(ptPosition:np.array, myMagnet:magnet) -> float:
    u0 = 1.26e-6 #[N*A**-2], permeability of free space. 
    # Here as a technicallity (doesn't matter tbh as is constant)
    r = ptPosition - myMagnet.position
    r_mag = np.sqrt(np.dot(r, r)) # magnitude of r; dist. be/en position & magnet
    try:
        B = u0 / (4*math.pi) * (3*np.dot(myMagnet.dipole, r)*r/r_mag**3 - myMagnet.dipole/r_mag**3)
    except ZeroDivisionError:
        B = 0 # set to 0 since don't care about this point
    return B

def totalMagneticFieldApprox(ptPosition:np.array, myMagnets:list) -> np.array:
    if len(myMagnets) == 0:
        print('no magnets defined, returning 0')
        return 0
    totalB = np.zeros((3,))
    for myMagnet in myMagnets:
        totalB += magnetFieldApprox(ptPosition, myMagnet)
    return totalB

def plotField(fieldMap:np.array) -> None:
    fig = go.Figure(data = go.Cone(
    x = np.concatenate((fieldMap[:,:,:8,0].flatten(), fieldMap[:,:,12:,0].flatten())), #fieldMap[:,:,:,0].flatten(),
    y = np.concatenate((fieldMap[:,:,:8,1].flatten(), fieldMap[:,:,12:,1].flatten())),#fieldMap[:,:,:,1].flatten(),
    z = np.concatenate((fieldMap[:,:,:8,2].flatten(), fieldMap[:,:,12:,2].flatten())),#fieldMap[:,:,:,2].flatten(),
    u = np.concatenate((fieldMap[:,:,:8,3].flatten(), fieldMap[:,:,12:,3].flatten())),#fieldMap[:,:,:,3].flatten(),
    v = np.concatenate((fieldMap[:,:,:8,4].flatten(), fieldMap[:,:,12:,4].flatten())),#fieldMap[:,:,:,4].flatten(),
    w = np.concatenate((fieldMap[:,:,:8,5].flatten(), fieldMap[:,:,12:,5].flatten())),#fieldMap[:,:,:,5].flatten(),
    sizeref=2,
    anchor="tip"))
    
    fig.update_layout(
      scene=dict(domain_x=[0, 1],
                 camera_eye=dict(x=3, y=3, z=3)))

    fig.show()
    return

# define magnets
myMagnets = list()
for pos in [np.array([1,1,0]), np.array([1,-1,0]), 
            np.array([-1,1,0]), np.array([-1,-1,0])]:
    myMagnets.append(magnet(pos, np.array([0,0,1])))
    
# define field to calculate over
fieldMap = np.zeros((21, 21, 21, 6)) # x_idx, y_idx, z_idx, np.array(x,y,z,Bx,By,Bz)
#not efficient given repeated position info but oh well
x_min = -2
y_min = -2
z_min = -2
x_max = 2
y_max = 2
z_max = 2

x_step = (x_max - x_min) / (fieldMap.shape[0]-1)
y_step = (y_max - y_min) / (fieldMap.shape[0]-1)
z_step = (z_max - z_min) / (fieldMap.shape[0]-1)
for i, x_pos in enumerate(np.arange(x_min, x_max + x_step, x_step)):
    for j, y_pos in enumerate(np.arange(y_min, y_max + y_step, y_step)):
        for k, z_pos in enumerate(np.arange(z_min, z_max + z_step, z_step)):
            ptPosition = np.array([x_pos, y_pos, z_pos])
            #print(ptPosition)
            #print(totalMagneticFieldApprox(ptPosition, myMagnets))
            fieldMap[i, j, k] = np.concatenate(
                (ptPosition, 
                 totalMagneticFieldApprox(ptPosition, myMagnets))
                )

plotField(fieldMap)